# Piloto Text2KG — Modelos até 3B (Colab T4)

**Configuração antes de rodar:**
- Runtime → Change runtime type → **T4 GPU**
- Adicione os secrets em Tools → Secrets:
  - `GITHUB_TOKEN` — acesso ao repositório privado
  - `HF_TOKEN` — token HuggingFace

**Modelos suportados (escolha na Célula 1):**
- `Qwen/Qwen2.5-3B-Instruct` — 3B, bfloat16, sem AWQ (padrão)
- `Qwen/Qwen2.5-1.5B-Instruct` — 1.5B, mais rápido
- `Qwen/Qwen2.5-3B-Instruct-AWQ` — 3B AWQ, se disponível no HF

**Tempo estimado:** ~1.5h para 100 frases × 2 ontologias × 3 condições.

In [ ]:
# Célula 1 — Configuração
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
HF_TOKEN     = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# --- Escolha o modelo ---
MODELO = "Qwen/Qwen2.5-3B-Instruct"   # altere aqui se quiser outro modelo
DTYPE  = "bfloat16"                     # T4 suporta bf16; use 'half' se der erro
PORT   = 8000
N_FRASES = 100  # Colab T4 single: use 100 frases para caber no tempo
FRASES_SEED = 42

REPO_URL  = f"https://{GITHUB_TOKEN}@github.com/fmsoares/llm-kg-ontology.git"
BENCH_URL = "https://github.com/nandana/Text2KGBench.git"

REPO_DIR   = "/content/llm-kg-ontology"
BENCH_DIR  = "/content/Text2KGBench"
RESULT_DIR = f"{REPO_DIR}/resultados/colab"
C2_DIR     = f"{REPO_DIR}/configs/c2"
VLLM_LOG   = "/content/vllm.log"

print(f"Modelo:  {MODELO}")
print(f"N_frases: {N_FRASES} por experimento")

In [ ]:
# Célula 2 — Clonar repositórios
import subprocess

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Falhou (rc={result.returncode}): {cmd}")
    return result

if not os.path.exists(REPO_DIR):
    run(f"git clone --depth=1 {REPO_URL} {REPO_DIR}")
else:
    run(f"git -C {REPO_DIR} pull")

if not os.path.exists(BENCH_DIR):
    run(f"git clone --depth=1 {BENCH_URL} {BENCH_DIR}")

print("Clones OK.")

In [ ]:
# Célula 3 — Instalar dependências
run("pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy pandas statsmodels")

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
print("Dependências OK.")

In [ ]:
# Célula 4 — Subir vLLM em segundo plano
# T4 single: tensor-parallel-size=1, sem AWQ se modelo for denso

vllm_cmd = (
    f"vllm serve {MODELO} "
    f"--port {PORT} "
    f"--dtype {DTYPE} "
    f"--tensor-parallel-size 1 "
    f"--max-model-len 4096 "
    f"--gpu-memory-utilization 0.90 "
    f"> {VLLM_LOG} 2>&1 &"
)
os.system(vllm_cmd)
print(f"vLLM iniciado (modelo={MODELO}).")
print("Aguarde ~60s para o modelo carregar.")

In [ ]:
# Célula 5 — Aguardar vLLM pronto
import time, urllib.request, urllib.error

VLLM_URL = f"http://localhost:{PORT}"
max_wait, intervalo, elapsed = 240, 10, 0

while elapsed < max_wait:
    try:
        urllib.request.urlopen(f"{VLLM_URL}/health", timeout=3)
        print(f"vLLM pronto após {elapsed}s.")
        break
    except (urllib.error.URLError, ConnectionRefusedError):
        print(f"  aguardando... ({elapsed}s)")
        time.sleep(intervalo)
        elapsed += intervalo
else:
    print("TIMEOUT. Log:")
    run(f"tail -40 {VLLM_LOG}")

In [ ]:
# Célula 6 — Gerar manifesto com caminhos do Colab
import json
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)

run(
    f"python {REPO_DIR}/scripts/gerar_manifesto.py "
    f"--modelo {MODELO} "
    f"--url http://localhost:{PORT}/v1 "
    f"--saida_dir {REPO_DIR}/configs/pipeline",
    cwd=REPO_DIR
)

piloto_path = f"{REPO_DIR}/configs/pipeline/manifesto_piloto_kaggle.json"
with open(piloto_path, encoding="utf-8") as f:
    piloto = json.load(f)

# Ajusta caminhos e frases_max para Colab (100 frases)
for exp in piloto:
    exp["saida"]      = f"{RESULT_DIR}/{os.path.basename(exp['saida'])}"
    exp["frases_max"] = N_FRASES
    exp["frases_seed"] = FRASES_SEED

piloto_colab_path = f"/content/manifesto_piloto_colab.json"
with open(piloto_colab_path, "w", encoding="utf-8") as f:
    json.dump(piloto, f, ensure_ascii=False, indent=2)

print(f"Manifesto com {len(piloto)} experimentos -> {piloto_colab_path}")

In [ ]:
# Célula 7 — Executar piloto

run(
    f"python -m t2kg.run "
    f"--manifesto {piloto_colab_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--skip_existentes "
    f"--verbose",
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
)
print("\nPiloto concluído.")

In [ ]:
# Célula 8 — Indução C2 (opcional — calibração D-16)
# Colab pode não ter tempo suficiente para ambas as ontologias.
# Ajuste para uma ontologia se necessário.

for onto in ["1_movie"]:
    print(f"\n--- Induzindo C2 para tekgen/{onto} ---")
    run(
        f"python {REPO_DIR}/scripts/induzir_c2.py "
        f"--dataset tekgen "
        f"--onto {onto} "
        f"--url http://localhost:{PORT}/v1 "
        f"--modelo_llm {MODELO} "
        f"--threshold 0.80 "
        f"--n_frases 50 "
        f"--seed 42 "
        f"--calibrar "
        f"--saida_dir {C2_DIR}",
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
    )

print("Indução C2 concluída.")

In [ ]:
# Célula 9 — Avaliar resultados

CSV_OUT = f"{RESULT_DIR}/tabela_colab.csv"

run(
    f"python {REPO_DIR}/scripts/avaliar_resultados.py "
    f"--manifesto {piloto_colab_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--csv {CSV_OUT}",
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
)

import pandas as pd
df = pd.read_csv(CSV_OUT)
print(df.to_string())

In [ ]:
# Célula 10 — Download dos resultados (alternativa ao git push)
# Baixa os arquivos de resultado para o computador local.

from google.colab import files
import glob

arquivos = glob.glob(f"{RESULT_DIR}/*.jsonl") + glob.glob(f"{RESULT_DIR}/*.csv")
arquivos += glob.glob(f"{C2_DIR}/*.json")

for arq in arquivos:
    print(f"Baixando: {arq}")
    files.download(arq)

In [ ]:
# Célula 11 — Commit e push (alternativa ao download)

GIT_EMAIL = "felipemoraissoares@gmail.com"
GIT_NAME  = "Felipe Morais (Colab)"

run(f"git -C {REPO_DIR} config user.email '{GIT_EMAIL}'")
run(f"git -C {REPO_DIR} config user.name '{GIT_NAME}'")
run(f"git -C {REPO_DIR} add resultados/colab/ configs/c2/")

modelo_slug = MODELO.split('/')[-1].lower().replace('-', '_')
commit_msg = f"resultados/colab: {modelo_slug}, {N_FRASES} frases, C0/C1/C1_oneshot"
run(f'git -C {REPO_DIR} commit -m "{commit_msg}" || echo "Nada para commitar"')
run(f"git -C {REPO_DIR} push")
print("Push concluído.")